# Heating Oil Contracts - Split by Keyword

Load `disaggregated_combined.csv`, filter to all `commodity_name` values
containing "HEATING" or "DIESEL", then split the contracts by keyword in
`market_and_exchange_names`:
- **Heating Oil / ULSD** - contains "HEATING OIL" or "ULSD" or "NY HARBOR"
- **Gasoil** - contains "GASOIL"
- **Jet** - contains "JET"
- **Biodiesel** - contains "BIODIESEL"
- **Other** - everything else

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)

# Filter to all commodity_name values containing HEATING or DIESEL
mask = (
    df["commodity_name"].str.contains("HEATING", case=False, na=False)
    | df["commodity_name"].str.contains("DIESEL", case=False, na=False)
)
ho_all = df[mask].copy()
print(f"Total rows: {len(ho_all):,}")
print(f"\ncommodity_name values:")
print(ho_all["commodity_name"].value_counts())

In [ ]:
# Get all unique (code, name) pairs
contracts = (
    ho_all[["cftc_contract_market_code", "market_and_exchange_names"]]
    .drop_duplicates()
    .sort_values(["cftc_contract_market_code", "market_and_exchange_names"])
    .reset_index(drop=True)
)

name = contracts["market_and_exchange_names"].str.upper()
has_ho = name.str.contains("HEATING OIL|ULSD|NY HARBOR", regex=True)
has_gasoil = name.str.contains("GASOIL", regex=False)
has_jet = name.str.contains("JET", regex=False)
has_biodiesel = name.str.contains("BIODIESEL", regex=False)

contracts["group"] = "Other"
contracts.loc[has_ho & ~has_gasoil & ~has_jet, "group"] = "Heating Oil / ULSD"
contracts.loc[has_gasoil & ~has_ho & ~has_jet, "group"] = "Gasoil"
contracts.loc[has_jet, "group"] = "Jet"
contracts.loc[has_biodiesel, "group"] = "Biodiesel"
# Mixed HO + Gasoil = spread between the two
contracts.loc[has_ho & has_gasoil, "group"] = "HO-Gasoil Spread"

print(f"Total unique (code, name) pairs: {len(contracts)}")
print(f"\nBy group:")
print(contracts["group"].value_counts())

## Heating Oil / ULSD Contracts

In [ ]:
ho = contracts[contracts["group"] == "Heating Oil / ULSD"].sort_values("cftc_contract_market_code")
print(f"HO rows: {len(ho)}  |  Unique codes: {ho['cftc_contract_market_code'].nunique()}")
ho

## Gasoil Contracts

In [ ]:
gasoil = contracts[contracts["group"] == "Gasoil"].sort_values("cftc_contract_market_code")
print(f"Gasoil rows: {len(gasoil)}  |  Unique codes: {gasoil['cftc_contract_market_code'].nunique()}")
gasoil

## HO-Gasoil Spread Contracts

In [ ]:
ho_gasoil = contracts[contracts["group"] == "HO-Gasoil Spread"].sort_values("cftc_contract_market_code")
print(f"HO-Gasoil Spread rows: {len(ho_gasoil)}  |  Unique codes: {ho_gasoil['cftc_contract_market_code'].nunique()}")
ho_gasoil

## Jet Contracts

In [ ]:
jet = contracts[contracts["group"] == "Jet"].sort_values("cftc_contract_market_code")
print(f"Jet rows: {len(jet)}  |  Unique codes: {jet['cftc_contract_market_code'].nunique()}")
jet

## Biodiesel Contracts

In [ ]:
biodiesel = contracts[contracts["group"] == "Biodiesel"].sort_values("cftc_contract_market_code")
print(f"Biodiesel rows: {len(biodiesel)}  |  Unique codes: {biodiesel['cftc_contract_market_code'].nunique()}")
biodiesel

## Other Contracts

In [ ]:
# Exclude codes already identified in other groups
known_codes = set(
    contracts[contracts["group"] != "Other"]["cftc_contract_market_code"]
)
other = contracts[
    (contracts["group"] == "Other")
    & (~contracts["cftc_contract_market_code"].isin(known_codes))
].sort_values("cftc_contract_market_code")
print(f"Other rows: {len(other)}  |  Unique codes: {other['cftc_contract_market_code'].nunique()}")
other